# Transpose & Inverse
**Topic:** Linear Algebra for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the transpose operation as flipping a matrix along its diagonal
- **Explain** what the inverse of a matrix means geometrically: it undoes the matrix's transformation
- **Interpret** the normal equations for linear regression, which combine both operations into a single formula

> **Tip:** Start by looking at the color-coded transpose widget and trace where the element in row 2, column 1 ends up after transposing. Then switch to the inverse view and multiply A by A⁻¹ to see the identity matrix appear.

---
## How we got here

In *04: Matrix Multiplication* we learned to multiply matrices and traced the shapes through a neural network forward pass. Transpose and inverse are two properties that arise when we ask "how do we reverse or rearrange this multiplication?" The normal equations for linear regression, the closed-form solution that finds optimal weights without gradient descent, are built from exactly these two operations.

---
## Why this matters for data science

The closed-form solution to linear regression is: **θ = (XᵀX)⁻¹Xᵀy**. This formula uses the transpose Xᵀ to rearrange the data matrix, and the inverse (XᵀX)⁻¹ to solve the resulting system. Understanding this formula is not required to use `LinearRegression().fit(X, y)`, but it explains why linear regression is exact (not iterative), why it fails when features are collinear (making XᵀX uninvertible), and why gradient descent is needed for models where no such closed form exists.

---
## Try it yourself

In [2]:
out = Output()

view_tg = widgets.ToggleButtons(
    options=["Transpose", "Inverse"],
    value="Transpose",
    style={"button_width": "130px"})

# Transpose: fixed 3×4 matrix with readable values
_A_T = np.array([[1.0, 4.0, 7.0, 10.0],
                 [2.0, 5.0, 8.0, 11.0],
                 [3.0, 6.0, 9.0, 12.0]])

# Inverse: four sliders for a 2×2 matrix
a00_sl = FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.5,
    description="A[0,0]:", style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"))
a01_sl = FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.5,
    description="A[0,1]:", style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"))
a10_sl = FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.5,
    description="A[1,0]:", style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"))
a11_sl = FloatSlider(value=4.0, min=-5.0, max=5.0, step=0.5,
    description="A[1,1]:", style={"description_width": "70px"},
    layout=widgets.Layout(width="320px"))

inv_ctrl = VBox([HBox([a00_sl, a01_sl]), HBox([a10_sl, a11_sl])])
ctrl_box = VBox([view_tg])

def render(change=None):
    view = view_tg.value
    with out:
        clear_output(wait=True)
        if view == "Transpose":
            AT = _A_T.T  # shape (4, 3)
            nr, nc = _A_T.shape
            # Shared color value = element value (same data = same color scale)
            fig = make_subplots(rows=1, cols=2,
                subplot_titles=[f"A  shape ({nr}×{nc})", f"Aᵀ  shape ({nc}×{nr})"])
            fig.add_trace(go.Heatmap(
                z=_A_T, colorscale="Blues", showscale=False,
                zmin=_A_T.min(), zmax=_A_T.max(),
                text=_A_T.astype(int), texttemplate="%{text}",
                x=[f"col {j}" for j in range(nc)],
                y=[f"row {i}" for i in range(nr)]),
                row=1, col=1)
            fig.add_trace(go.Heatmap(
                z=AT, colorscale="Blues", showscale=False,
                zmin=_A_T.min(), zmax=_A_T.max(),
                text=AT.astype(int), texttemplate="%{text}",
                x=[f"col {j}" for j in range(nr)],
                y=[f"row {i}" for i in range(nc)]),
                row=1, col=2)
            fig.update_layout(
                title=dict(
                    text="Transpose: A[i,j] moves to Aᵀ[j,i] — rows become columns",
                    font=dict(size=FONT["size_title"], family=FONT["family"])),
                paper_bgcolor=PALETTE["background"], height=360,
                margin=dict(t=70, b=10, l=10, r=10))
            display(go.FigureWidget(fig))
            display(HTML(
                f'<div style="font-family:{FONT["family"]}; font-size:14px; '
                f'padding:8px 18px; background:#f8f9fa; border-radius:6px; margin-top:4px;">'
                f'A shape: <b>({nr}, {nc})</b>&emsp;→&emsp;Aᵀ shape: <b>({nc}, {nr})</b>'
                f'&emsp;|&emsp;e.g. A[2,0] = {_A_T[2,0]:.0f} → Aᵀ[0,2] = {AT[0,2]:.0f}'
                f'</div>'))

        else:  # Inverse
            A = np.array([[a00_sl.value, a01_sl.value],
                          [a10_sl.value, a11_sl.value]])
            det = float(np.linalg.det(A))
            invertible = abs(det) > 1e-6

            if invertible:
                A_inv = np.linalg.inv(A)
                A_at_Ainv = A @ A_inv
                panels = [("A", A, "Blues"),
                          ("A⁻¹", A_inv, "Oranges"),
                          ("A × A⁻¹ (should be I)", A_at_Ainv, "Greens")]
                fig = make_subplots(rows=1, cols=3,
                    subplot_titles=[p[0] for p in panels])
                for col, (_, M, cs) in enumerate(panels, 1):
                    fig.add_trace(go.Heatmap(z=M, colorscale=cs, showscale=False,
                        zmid=0 if cs == "Greens" else None,
                        text=M.round(4), texttemplate="%{text:.3f}"),
                        row=1, col=col)
                status_html = (
                    f'det(A) = {det:.4f}  →  <b style="color:#2d9c5f">Matrix is invertible</b>'
                    f'&emsp;|&emsp;A × A⁻¹ ≈ I (identity matrix)')
            else:
                fig = make_subplots(rows=1, cols=1)
                fig.add_trace(go.Heatmap(z=A, colorscale="Blues", showscale=False,
                    text=A.round(2), texttemplate="%{text}"))
                status_html = (
                    f'det(A) = {det:.6f}  →  '
                    f'<b style="color:#d93025">Not invertible — determinant is zero</b>'
                    f'&emsp;(rows are linearly dependent)')

            fig.update_layout(
                title=dict(
                    text=f"Inverse: det(A) = {det:.4f}  |  A × A⁻¹ = I",
                    font=dict(size=FONT["size_title"], family=FONT["family"])),
                paper_bgcolor=PALETTE["background"], height=310,
                margin=dict(t=65, b=10, l=10, r=10))
            display(go.FigureWidget(fig))
            display(HTML(
                f'<div style="font-family:{FONT["family"]}; font-size:14px; '
                f'padding:8px 18px; background:#f8f9fa; border-radius:6px; margin-top:4px;">'
                + status_html + '</div>'))

def on_view_change(change):
    if view_tg.value == "Transpose":
        ctrl_box.children = [view_tg, out]
    else:
        ctrl_box.children = [view_tg, inv_ctrl, out]
    render()

view_tg.observe(on_view_change, names="value")
for sl in [a00_sl, a01_sl, a10_sl, a11_sl]:
    sl.observe(render, names="value")

ctrl_box.children = [view_tg, out]
display(ctrl_box)
render()

---
## What's happening?

**Transpose** rearranges a matrix by swapping rows and columns. If **A** has shape (m, n), then its transpose **Aᵀ** has shape (n, m). Element A[i, j] moves to position Aᵀ[j, i].

$$\left(\mathbf{A}^\top\right)_{ij} = A_{ji}$$

Visually: the matrix is "reflected" along its diagonal (the top-left to bottom-right line).

**Inverse** of a square matrix **A** is the matrix **A⁻¹** such that multiplying them gives the identity matrix **I**, a matrix with 1s on the diagonal and 0s everywhere else:

$$\mathbf{A} \mathbf{A}^{-1} = \mathbf{A}^{-1} \mathbf{A} = \mathbf{I}$$

The identity matrix **I** plays the same role in matrix multiplication that the number 1 plays in regular multiplication.

### Not all matrices have an inverse

A matrix is invertible only if it is square (same number of rows and columns) and its determinant is non-zero. When two features in a dataset are perfectly correlated, **XᵀX** becomes non-invertible. This is why multicollinearity breaks the normal equations for linear regression.

| Operation | Notation | Shape change | Condition |
|-----------|----------|-------------|-----------|
| Transpose | Aᵀ | (m,n) → (n,m) | Always defined |
| Inverse | A⁻¹ | (n,n) → (n,n) | Only if A is square and det(A) ≠ 0 |
| Pseudo-inverse | A⁺ | (m,n) → (n,m) | Always defined; generalizes inverse to non-square |

### The normal equations: where both appear

The normal equations give the exact optimal weights for linear regression:

$$\boldsymbol{\theta} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

- **Xᵀ** rearranges the data so features align for the matrix multiply
- **XᵀX** produces a square, symmetric matrix (p × p where p is the number of features)
- **(XᵀX)⁻¹** inverts it to solve the system
- **Xᵀy** is the target vector projected onto the feature space

The result **θ** is the vector of optimal weights that minimizes MSE exactly, no iteration needed.

Return to the widget and verify that for a 2×2 matrix, A × A⁻¹ produces [[1,0],[0,1]] regardless of what A is (as long as it is invertible).

### Transpose and inverse in ML formulas

| Operation | Notation | ML formula where it appears | What it achieves |
|-----------|----------|-----------------------------|-----------------|
| Transpose | Xᵀ | Normal equations: θ = (XᵀX)⁻¹Xᵀy | Aligns feature matrix for inner product |
| Inverse | (XᵀX)⁻¹ | Normal equations | Solves the system of equations exactly |
| Pseudo-inverse | X⁺ | Over/under-determined least squares | Generalizes inverse to non-square X |
| Transpose in backprop | Wᵀ | δₗ = Wᵀ δₗ₊₁ (error propagation) | Routes gradients backward through the layer |

---
## Key takeaway

> **The transpose swaps rows and columns; the inverse undoes a matrix's transformation; together they produce the normal equations θ = (XᵀX)⁻¹Xᵀy, the exact closed-form solution to linear regression that no iteration can improve upon.**

---
*Next up: Norms & Distances — how to measure the size of a vector and the distance between two points, which directly drives regularization in Ridge and Lasso regression*